# Retirement Monte Carlo — Baseline Template

**Student Name**: [Your Name]

**Project**: LFC MarketView Retirement Monte Carlo Planner

**Goal**: Simulate 10,000 possible futures for a monthly investor and measure the range of retirement outcomes

---

## Structure
1. Compounding & Deterministic Growth (Week 1)
2. Monte Carlo Simulation (Week 2)
3. Analysis & Portfolio Comparison (Week 3)

---

## Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline
print('Libraries loaded')

---
## Week 1: Compounding & Deterministic Growth

**Goal**: Grow a portfolio month by month with a fixed return (no randomness yet)

In [ ]:
# Investor parameters
START_AGE            = 25
RETIREMENT_AGE       = 65
MONTHLY_CONTRIBUTION = 300      # € invested every month
YEARLY_RETURN        = 0.06     # 6% per year, fixed for now

YEARS  = RETIREMENT_AGE - START_AGE
MONTHS = YEARS * 12
monthly_return = YEARLY_RETURN / 12

print(f'Investing €{MONTHLY_CONTRIBUTION}/month for {YEARS} years ({MONTHS} months)')
print(f'Monthly return: {monthly_return:.4%}')

In [ ]:
# Grow the balance month by month
balance = 0
balances = []

for month in range(MONTHS):
    balance = balance * (1 + monthly_return) + MONTHLY_CONTRIBUTION
    balances.append(balance)

total_contributed = MONTHLY_CONTRIBUTION * MONTHS
print(f'Final balance:     €{balance:,.0f}')
print(f'Total contributed: €{total_contributed:,.0f}')

# Plot balance over time
ages = START_AGE + np.arange(1, MONTHS + 1) / 12
plt.figure(figsize=(14, 5))
plt.plot(ages, balances, linewidth=2, label='Portfolio value')
plt.plot(ages, MONTHLY_CONTRIBUTION * np.arange(1, MONTHS + 1),
         linestyle='--', label='Money contributed')
plt.title('Deterministic Growth of €300/month at 6%')
plt.xlabel('Age')
plt.ylabel('Value (€)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# TODO (Student): Turn the loop above into a function
#
# def grow(monthly_contribution, yearly_return, years):
#     ...
#     return final_balance
#
# Then use it to:
#   1. Split the final balance into 'contributed' vs 'growth from returns'
#   2. Compare starting at age 25 vs age 35 (same retirement age)
#   3. Try different returns: 3%, 6%, 9%
#
# Hint: growth = final_balance - total_contributed

print('Deterministic analysis: Student work here')

---
## Week 2: Monte Carlo Simulation

**Goal**: Replace the fixed return with random returns and simulate thousands of possible futures

In [ ]:
# Asset assumptions (yearly)
MU_STOCKS,  VOL_STOCKS = 0.07, 0.15
MU_BONDS,   VOL_BONDS  = 0.03, 0.05
CORRELATION            = 0.2

N_SIMULATIONS = 10_000
SEED          = 42   # fixes the random numbers so results are reproducible


def portfolio_params(w_stocks):
    """Yearly expected return and volatility of a stock/bond mix."""
    w_bonds = 1 - w_stocks
    mu = w_stocks * MU_STOCKS + w_bonds * MU_BONDS
    var = (w_stocks**2 * VOL_STOCKS**2
           + w_bonds**2 * VOL_BONDS**2
           + 2 * w_stocks * w_bonds * CORRELATION * VOL_STOCKS * VOL_BONDS)
    return mu, np.sqrt(var)


mu_p, vol_p = portfolio_params(0.60)
print(f'60/40 portfolio: expected return {mu_p:.2%}, volatility {vol_p:.2%} per year')

In [ ]:
# Simulate ONE random path with a for-loop
rng = np.random.default_rng(SEED)
monthly_mu  = mu_p / 12
monthly_vol = vol_p / np.sqrt(12)

balance = 0
one_path = []
for month in range(MONTHS):
    r = rng.normal(monthly_mu, monthly_vol)   # random return this month
    balance = balance * (1 + r) + MONTHLY_CONTRIBUTION
    one_path.append(balance)

plt.figure(figsize=(14, 5))
plt.plot(ages, one_path, linewidth=1.5, label='One random path')
plt.plot(ages, balances, linestyle='--', label='Deterministic (6%)')
plt.title('One Simulated Future vs Fixed Return')
plt.xlabel('Age')
plt.ylabel('Value (€)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
print(f'Final balance on this path: €{balance:,.0f}')

In [ ]:
# Simulate ALL paths at once with NumPy (vectorized: no loop over simulations)
def simulate(w_stocks, n_sims=N_SIMULATIONS, seed=SEED):
    """Returns an array of shape (n_sims, MONTHS) with the balance of every path."""
    mu, vol = portfolio_params(w_stocks)
    rng = np.random.default_rng(seed)
    returns = rng.normal(mu / 12, vol / np.sqrt(12), size=(n_sims, MONTHS))

    paths = np.zeros((n_sims, MONTHS))
    balance = np.zeros(n_sims)
    for m in range(MONTHS):            # loop over months only, all paths updated at once
        balance = balance * (1 + returns[:, m]) + MONTHLY_CONTRIBUTION
        paths[:, m] = balance
    return paths


paths = simulate(0.60)
final_balances = paths[:, -1]
print(f'Simulated paths: {paths.shape[0]:,} | Months per path: {paths.shape[1]}')

In [ ]:
# Fan chart: all paths in grey, median and 10th/90th percentiles highlighted
p10, p50, p90 = np.percentile(paths, [10, 50, 90], axis=0)

plt.figure(figsize=(14, 6))
plt.plot(ages, paths[:200].T, color='grey', alpha=0.05)   # first 200 paths
plt.fill_between(ages, p10, p90, alpha=0.3, label='10th–90th percentile')
plt.plot(ages, p50, linewidth=2.5, label='Median')
plt.title('Monte Carlo: 60/40 Portfolio, €300/month')
plt.xlabel('Age')
plt.ylabel('Value (€)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# TODO (Student): Histogram of final balances
#
# - Plot a histogram of final_balances (try bins=100)
# - Draw vertical lines at the 10th, 50th and 90th percentiles
# - Print the three percentile values formatted as € with commas
# - Also print the MEAN. Is it higher or lower than the median? Why?
#
# Hint: plt.hist(), plt.axvline(), np.percentile(final_balances, 10)

print('Final wealth distribution: Student work here')

---
## Week 3: Analysis & Portfolio Comparison

**TODO**: Turn the simulations into decisions

In [ ]:
# TODO (Student): Probability of reaching a target
#
# TARGET = 500_000
# prob = (final_balances >= TARGET).mean()
#
# - Print the probability as a percentage
# - Try 3 different targets: €300k, €500k, €1M

print('Target probability: Student work here')

In [ ]:
# TODO (Student): Compare 3 portfolios
#
# PORTFOLIOS = {'Conservative': 0.30, 'Balanced': 0.60, 'Aggressive': 1.00}
#
# For each portfolio:
#   - paths = simulate(w_stocks)
#   - final = paths[:, -1]
#   - store median, 10th percentile, 90th percentile, P(€500k)
#
# Then plot the 3 histograms on the same chart (use alpha=0.5)

print('Portfolio comparison: Student work here')

In [ ]:
# TODO (Student): Adjust for inflation
#
# €1 in 40 years will buy much less than €1 today.
# INFLATION = 0.02
# real_value = nominal_value / (1 + INFLATION) ** YEARS
#
# - Calculate the real median for each portfolio
# - Plot nominal vs real median growth over time for the Balanced portfolio

print('Inflation adjustment: Student work here')

In [ ]:
# TODO (Student): Comparison table (fill in your results)
comparison = pd.DataFrame({
    'Metric':        ['Median Final', 'Bad Case (P10)', 'Good Case (P90)', 'P(€500k)', 'Real Median'],
    'Conservative':  ['TBD', 'TBD', 'TBD', 'TBD', 'TBD'],
    'Balanced':      ['TBD', 'TBD', 'TBD', 'TBD', 'TBD'],
    'Aggressive':    ['TBD', 'TBD', 'TBD', 'TBD', 'TBD'],
})
comparison

---
## Conclusions

**TODO**: Write your analysis here

### Which Portfolio Would You Choose at 25? At 55?
[Answer]

### Why the Median and Not the Average?
[Answer]

### What Surprised You?
[Answer]

### Limitations of the Model
[Answer]

### Potential Improvements
[Answer]

---
**Completed by**: [Your Name] | **Date**: [Date]